# 台股每日更新 Pipeline (Colab Pro+)

取代 Cloud Run 的每日更新任務。排程於每個交易日 19:00 TWN 執行。

**流程：**
1. 掛載 Google Drive（資料永久存儲）
2. Clone/Pull GitHub 最新代碼
3. 檢查是否為交易日
4. 從 Drive 複製資料到本地（加快 I/O）
5. 執行每日更新（close matrix + RS + screener + forward returns）
6. 建構 20 維技術特徵
7. 建構 20 維 Golden Templates
8. 將結果存回 Google Drive
9. 發送 Telegram 通知（SUCCESS / WARNING / ABORT）

## Cell 1: 環境設定 + 掛載 Google Drive

In [ ]:
import os
import time
import logging

# 設定 logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
logger = logging.getLogger('colab.daily_update')

# 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Google Drive 上的資料目錄
DRIVE_DATA_DIR = '/content/drive/MyDrive/Stock/data'
DRIVE_FEATURES_DIR = os.path.join(DRIVE_DATA_DIR, 'pattern_data', 'features')

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(DRIVE_FEATURES_DIR, exist_ok=True)

print(f'Drive 資料目錄: {DRIVE_DATA_DIR}')
print(f'Drive 特徵目錄: {DRIVE_FEATURES_DIR}')

# 列出現有檔案
if os.path.exists(DRIVE_DATA_DIR):
    files = os.listdir(DRIVE_DATA_DIR)
    print(f'\nDrive 現有檔案 ({len(files)}):');
    for f in sorted(files)[:20]:
        path = os.path.join(DRIVE_DATA_DIR, f)
        size = os.path.getsize(path) / (1024*1024) if os.path.isfile(path) else 0
        print(f'  {f} ({size:.1f} MB)' if size > 0 else f'  {f}/')

## Cell 2: Clone/Pull 代碼 + 安裝依賴

In [ ]:
REPO_DIR = '/content/Stock'

# Clone 或 Pull 最新代碼
if os.path.exists(REPO_DIR):
    print('Repo 已存在，執行 git pull...')
    !cd /content/Stock && git pull --ff-only
else:
    print('首次 Clone repo...')
    !git clone https://github.com/ZQCHUNG/Stock.git /content/Stock

# 安裝依賴（靜默模式）
print('\n安裝 Python 依賴...')
!pip install yfinance pandas numpy pyarrow requests feedparser lxml twstock -q

# 設定 Python path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f'\n工作目錄: {os.getcwd()}')
print(f'Python path 已加入: {REPO_DIR}')

## Cell 3: 檢查是否為交易日

In [ ]:
from datetime import datetime, date, timedelta
import pytz

tw_tz = pytz.timezone('Asia/Taipei')
now_tw = datetime.now(tw_tz)
today = now_tw.date()
weekday = today.weekday()

IS_TRADING_DAY = weekday < 5  # 週一至週五

print(f'台灣時間: {now_tw.strftime("%Y-%m-%d %H:%M:%S %A")}')
print(f'交易日: {"是" if IS_TRADING_DAY else "否（週末，跳過後續步驟）"}')

if not IS_TRADING_DAY:
    print('\n今天不是交易日，Notebook 到此結束。')
    # 排程模式下會自然結束，不需要 raise

## Cell 4: 從 Drive 複製資料到本地

本地磁碟 I/O 比 Drive 快很多，先複製到 `/content/Stock/data/` 再處理。

In [ ]:
import shutil

if not IS_TRADING_DAY:
    print('非交易日，跳過。')
else:
    # 需要從 Drive 複製到本地的檔案
    FILES_TO_COPY = [
        # (Drive 相對路徑, 本地相對路徑)
        ('pit_close_matrix.parquet', 'data/pit_close_matrix.parquet'),
        ('pit_rs_matrix.parquet', 'data/pit_rs_matrix.parquet'),
        ('pit_rs_percentile.parquet', 'data/pit_rs_percentile.parquet'),
        ('screener.db', 'data/screener.db'),
        ('market_data.db', 'data/market_data.db'),
        ('pattern_data/features/features_all.parquet', 'data/pattern_data/features/features_all.parquet'),
        ('pattern_data/features/forward_returns.parquet', 'data/pattern_data/features/forward_returns.parquet'),
        ('pattern_data/features/price_cache.parquet', 'data/pattern_data/features/price_cache.parquet'),
        ('pattern_data/features/feature_metadata.json', 'data/pattern_data/features/feature_metadata.json'),
        ('pattern_data/features/daily_features.parquet', 'data/pattern_data/features/daily_features.parquet'),
        ('pattern_data/features/daily_feature_metadata.json', 'data/pattern_data/features/daily_feature_metadata.json'),
        ('pattern_data/features/golden_templates_20d.parquet', 'data/pattern_data/features/golden_templates_20d.parquet'),
        ('pattern_data/features/golden_templates_20d_norms.npy', 'data/pattern_data/features/golden_templates_20d_norms.npy'),
        ('pattern_data/features/golden_templates_20d_metadata.json', 'data/pattern_data/features/golden_templates_20d_metadata.json'),
    ]

    copied = 0
    missing = 0

    for drive_rel, local_rel in FILES_TO_COPY:
        src = os.path.join(DRIVE_DATA_DIR, drive_rel)
        dst = os.path.join(REPO_DIR, local_rel)

        if not os.path.exists(src):
            print(f'  [缺] {drive_rel}')
            missing += 1
            continue

        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f'  [OK] {drive_rel} ({size_mb:.1f} MB)')
        copied += 1

    print(f'\n複製完成: {copied} 檔案成功, {missing} 檔案缺失')

    if missing > 0 and not os.path.exists(os.path.join(REPO_DIR, 'data/pit_close_matrix.parquet')):
        print('\n⚠️ 警告: pit_close_matrix.parquet 不存在！')
        print('請先上傳初始資料到 Google Drive:')
        print(f'  {DRIVE_DATA_DIR}/pit_close_matrix.parquet')

## Cell 5: 執行每日更新 Pipeline

9 步流程：close matrix → sanitizer → RS → screener → forward returns → signals → trailing stops → auto-sim → review

In [ ]:
daily_result = None

if not IS_TRADING_DAY:
    print('非交易日，跳過。')
else:
    t0 = time.time()
    print('=' * 60)
    print('Step 1: 每日更新 Pipeline')
    print('=' * 60)

    try:
        from data.daily_update import run_daily_update
        daily_result = run_daily_update()

        elapsed = time.time() - t0
        print(f'\n每日更新完成 ({elapsed:.1f}s)')
        print(f'\n結果摘要:')
        for key, val in daily_result.items():
            if key not in ('timestamp',):
                print(f'  {key}: {val}')
    except Exception as e:
        elapsed = time.time() - t0
        print(f'\n每日更新失敗 ({elapsed:.1f}s): {e}')
        daily_result = {'error': str(e), 'total_elapsed_s': elapsed}
        import traceback
        traceback.print_exc()

## Cell 6: 建構 20 維技術特徵

In [ ]:
features_result = None

if not IS_TRADING_DAY:
    print('非交易日，跳過。')
else:
    t0 = time.time()
    print('=' * 60)
    print('Step 2a: 建構 20 維每日技術特徵')
    print('=' * 60)

    try:
        from data.build_daily_features import build_daily_features
        features_result = build_daily_features()

        elapsed = time.time() - t0
        print(f'\n特徵建構完成 ({elapsed:.1f}s)')
        print(f'  總行數: {features_result.get("total_rows", "?"):,}')
        print(f'  股票數: {features_result.get("stock_count", "?")}')
        print(f'  特徵數: {features_result.get("total_features", "?")}')
    except Exception as e:
        elapsed = time.time() - t0
        print(f'\n特徵建構失敗 ({elapsed:.1f}s): {e}')
        features_result = {'error': str(e)}
        import traceback
        traceback.print_exc()

## Cell 7: 建構 20 維 Golden Templates

In [ ]:
templates_result = None

if not IS_TRADING_DAY:
    print('非交易日，跳過。')
else:
    t0 = time.time()
    print('=' * 60)
    print('Step 2b: 建構 20 維 Golden Templates')
    print('=' * 60)

    try:
        from data.build_golden_templates_20d import build_golden_templates_20d
        templates_result = build_golden_templates_20d()

        elapsed = time.time() - t0
        print(f'\nGolden Templates 建構完成 ({elapsed:.1f}s)')
        print(f'  模板數: {templates_result.get("after_cap", "?")}')
        print(f'  股票數: {templates_result.get("stocks_represented", "?")}')
    except Exception as e:
        elapsed = time.time() - t0
        print(f'\nGolden Templates 建構失敗 ({elapsed:.1f}s): {e}')
        templates_result = {'error': str(e)}
        import traceback
        traceback.print_exc()

## Cell 8: 結果存回 Google Drive

將更新後的所有資料檔複製回 Drive，確保下次執行時有最新資料。

In [ ]:
saved_count = 0

if not IS_TRADING_DAY:
    print('非交易日，跳過。')
else:
    print('=' * 60)
    print('Step 3: 存回 Google Drive')
    print('=' * 60)

    # 需要存回 Drive 的檔案（與 Cell 4 對應）
    FILES_TO_SAVE = [
        ('data/pit_close_matrix.parquet', 'pit_close_matrix.parquet'),
        ('data/pit_rs_matrix.parquet', 'pit_rs_matrix.parquet'),
        ('data/pit_rs_percentile.parquet', 'pit_rs_percentile.parquet'),
        ('data/screener.db', 'screener.db'),
        ('data/market_data.db', 'market_data.db'),
        ('data/pattern_data/features/daily_features.parquet', 'pattern_data/features/daily_features.parquet'),
        ('data/pattern_data/features/daily_feature_metadata.json', 'pattern_data/features/daily_feature_metadata.json'),
        ('data/pattern_data/features/forward_returns.parquet', 'pattern_data/features/forward_returns.parquet'),
        ('data/pattern_data/features/price_cache.parquet', 'pattern_data/features/price_cache.parquet'),
        ('data/pattern_data/features/feature_metadata.json', 'pattern_data/features/feature_metadata.json'),
        ('data/pattern_data/features/golden_templates_20d.parquet', 'pattern_data/features/golden_templates_20d.parquet'),
        ('data/pattern_data/features/golden_templates_20d_norms.npy', 'pattern_data/features/golden_templates_20d_norms.npy'),
        ('data/pattern_data/features/golden_templates_20d_metadata.json', 'pattern_data/features/golden_templates_20d_metadata.json'),
        # 額外輸出
        ('data/self_healed_events.json', 'self_healed_events.json'),
    ]

    # 另外保留每日快照（按日期歸檔）
    today_str = now_tw.strftime('%Y-%m-%d')
    daily_archive_dir = os.path.join(DRIVE_DATA_DIR, 'daily_archive', today_str)
    os.makedirs(daily_archive_dir, exist_ok=True)

    for local_rel, drive_rel in FILES_TO_SAVE:
        src = os.path.join(REPO_DIR, local_rel)
        if not os.path.exists(src):
            continue

        # 存到 latest 位置
        dst_latest = os.path.join(DRIVE_DATA_DIR, drive_rel)
        os.makedirs(os.path.dirname(dst_latest), exist_ok=True)
        shutil.copy2(src, dst_latest)

        # 也存一份到日期歸檔（核心檔案）
        if drive_rel.endswith('.parquet') or drive_rel.endswith('.db'):
            dst_archive = os.path.join(daily_archive_dir, os.path.basename(drive_rel))
            shutil.copy2(src, dst_archive)

        size_mb = os.path.getsize(src) / (1024 * 1024)
        print(f'  [OK] {drive_rel} ({size_mb:.1f} MB)')
        saved_count += 1

    print(f'\n存回完成: {saved_count} 檔案')
    print(f'每日歸檔: {daily_archive_dir}')

## Cell 9: 發送 Telegram 通知

三級警報：SUCCESS / WARNING / ABORT

In [ ]:
import requests as req_lib

TG_TOKEN = '8612197310:AAF4qbJvTz549ieQvbQbC_wkB4GmtmAEAdk'
TG_CHAT_ID = '8168993656'

if not IS_TRADING_DAY:
    print('非交易日，跳過通知。')
else:
    print('=' * 60)
    print('Step 4: Telegram 通知')
    print('=' * 60)

    # --- 判斷警報等級 ---
    alert_level = 'SUCCESS'
    issues = []

    # 取得 pipeline 結果
    stock_count = 0
    total_stocks = 0
    coverage_pct = 0
    total_elapsed_s = 0

    if daily_result and 'error' not in daily_result:
        cm_info = daily_result.get('close_matrix', {})
        stock_count = daily_result.get('stock_count', cm_info.get('total_stocks', 0))
        total_stocks = daily_result.get('total_stocks', cm_info.get('total_stocks', 0))
        coverage_pct = daily_result.get('coverage_pct', 0)
        total_elapsed_s = daily_result.get('total_elapsed_s', 0)
    elif daily_result and 'error' in daily_result:
        alert_level = 'ABORT'
        issues.append(f'每日更新失敗: {daily_result["error"][:120]}')

    if features_result and 'error' in features_result:
        if alert_level != 'ABORT':
            alert_level = 'WARNING'
        issues.append(f'特徵建構失敗: {features_result["error"][:120]}')

    if templates_result and 'error' in templates_result:
        if alert_level != 'ABORT':
            alert_level = 'WARNING'
        issues.append(f'Golden Templates 建構失敗: {templates_result["error"][:120]}')

    # 覆蓋率檢查
    if total_stocks > 0 and coverage_pct < 90 and alert_level == 'SUCCESS':
        alert_level = 'WARNING'
        issues.append(f'低覆蓋率: {stock_count}/{total_stocks} ({coverage_pct}%)')

    # 執行時間檢查（> 30 分鐘）
    if total_elapsed_s > 1800 and alert_level == 'SUCCESS':
        alert_level = 'WARNING'
        issues.append(f'執行時間過長: {total_elapsed_s/60:.1f} 分鐘')

    # --- 組裝訊息 ---
    level_icon = {'SUCCESS': 'OK', 'WARNING': 'WARNING', 'ABORT': 'ABORT'}
    prefix = level_icon.get(alert_level, 'INFO')

    if total_stocks > 0:
        coverage_str = f'{stock_count:,}/{total_stocks:,} ({coverage_pct}%)'
    elif stock_count > 0:
        coverage_str = f'{stock_count:,} stocks'
    else:
        coverage_str = '? stocks'

    elapsed_min = total_elapsed_s / 60

    # 特徵結果摘要
    feat_str = ''
    if features_result and 'error' not in features_result:
        feat_str = f'\nFeatures: {features_result.get("stock_count", "?")} stocks, {features_result.get("total_rows", "?"):,} rows'

    tpl_str = ''
    if templates_result and 'error' not in templates_result:
        tpl_str = f'\nTemplates: {templates_result.get("after_cap", "?")} templates'

    if alert_level == 'SUCCESS':
        msg = f'*[{prefix}]* Colab Daily Update\n{coverage_str}, {elapsed_min:.1f} min{feat_str}{tpl_str}\nSaved: {saved_count} files to Drive'
    elif alert_level == 'WARNING':
        issue_list = '\n'.join(f'  - {i}' for i in issues)
        msg = f'*[{prefix}]* Colab Daily Update (warnings):\n{issue_list}\n{coverage_str}, {elapsed_min:.1f} min{feat_str}{tpl_str}'
    else:  # ABORT
        reason = issues[0] if issues else 'Unknown error'
        msg = f'*[{prefix}]* Colab Daily Update ABORTED\n{reason}'

    # --- 發送 ---
    try:
        resp = req_lib.post(
            f'https://api.telegram.org/bot{TG_TOKEN}/sendMessage',
            json={'chat_id': TG_CHAT_ID, 'text': msg, 'parse_mode': 'Markdown'},
            timeout=10,
        )
        if resp.ok:
            print(f'Telegram 通知已發送 [{alert_level}]')
        else:
            print(f'Telegram 發送失敗: {resp.status_code} {resp.text[:200]}')
    except Exception as e:
        print(f'Telegram 發送異常: {e}')

    print(f'\n訊息內容:\n{msg}')

## Cell 10: 總結

顯示本次執行的完整摘要。

In [ ]:
print('=' * 60)
print('Colab Daily Update — 執行摘要')
print('=' * 60)
print(f'日期: {now_tw.strftime("%Y-%m-%d %H:%M:%S") if IS_TRADING_DAY else "非交易日"}')
print(f'交易日: {"是" if IS_TRADING_DAY else "否"}')

if IS_TRADING_DAY:
    print(f'\n--- 每日更新 ---')
    if daily_result:
        if 'error' in daily_result:
            print(f'  狀態: 失敗 ({daily_result["error"][:80]})')
        else:
            cm = daily_result.get('close_matrix', {})
            print(f'  Close Matrix: +{cm.get("new_dates", "?")} 天, {cm.get("total_stocks", "?")} 檔')
            rs = daily_result.get('rs_matrices', {})
            print(f'  RS 矩陣: {rs.get("rs_stocks", "?")} 檔 ({rs.get("elapsed_s", "?"):.1f}s)' if 'rs_stocks' in rs else f'  RS 矩陣: {rs}')
            sc = daily_result.get('screener', {})
            print(f'  Screener: {sc.get("stocks_updated", "?")} 檔更新' if 'stocks_updated' in sc else f'  Screener: {sc}')
            fwd = daily_result.get('forward_returns', {})
            print(f'  Forward Returns: filled {fwd.get("filled", "?")} NaN' if 'filled' in fwd else f'  Forward Returns: {fwd}')
            san = daily_result.get('sanitizer', {})
            print(f'  Sanitizer: {san.get("anomalies_found", 0)} anomalies, {san.get("healed", 0)} healed')
            print(f'  總耗時: {daily_result.get("total_elapsed_s", 0):.1f}s')

    print(f'\n--- 特徵建構 ---')
    if features_result:
        if 'error' in features_result:
            print(f'  狀態: 失敗 ({features_result["error"][:80]})')
        else:
            print(f'  行數: {features_result.get("total_rows", "?"):,}')
            print(f'  股票: {features_result.get("stock_count", "?")}')
            print(f'  耗時: {features_result.get("elapsed_s", "?")}s')

    print(f'\n--- Golden Templates ---')
    if templates_result:
        if 'error' in templates_result:
            print(f'  狀態: 失敗 ({templates_result["error"][:80]})')
        else:
            print(f'  模板數: {templates_result.get("after_cap", "?")}')
            print(f'  股票數: {templates_result.get("stocks_represented", "?")}')

    print(f'\n--- Drive 存檔 ---')
    print(f'  已存: {saved_count} 檔案')
    print(f'  位置: {DRIVE_DATA_DIR}')

print(f'\n完成。')